# Install

`onceover` is a Python package with a Rust core. Binary wheels are the normal
path; building from source needs a Rust toolchain.

```bash
pip install onceover
```

The package requires CPython 3.10+ and Polars 2.x. Wheels cover Linux x86_64
and aarch64, macOS Intel and Apple silicon, and Windows x86_64. The source
distribution additionally needs Rust and a C compiler — the same setup used
for development, below.

## Verify the installation

The first profile is a thirty-second check: `import onceover` loads the
compiled extension, and a tiny frame exercises the whole pipeline.

A word about scale before you try something bigger: this profiler accepts
in-memory DataFrames with millions of rows and hundreds of columns. Column
summaries and integrity checks always examine every row; relationship
discovery is bounded — that is a feature, and
[](coverage.ipynb) explains the contract in full.

In [ ]:
import polars as pl
import onceover

print(onceover.__version__)

df = pl.DataFrame({"id": [1, 2, 3], "value": [0.5, 1.5, None]})
report = onceover.profile(df, budget="quick")
print(report.overview["headline"])

## Optional dependencies

Two extras install what the surrounding tooling needs:

| Extra | Installs | Used by |
|---|---|---|
| `onceover[notebook]` | altair, great-tables, vl-convert, numpy, jupyterlab | the analysis notebook, charts, tables |
| `onceover[benchmark]` | numpy, psutil | the scale benchmark |

```bash
pip install 'onceover[notebook]'
```

## Development installation

Development builds the Rust core and binds it to a virtual environment:

```bash
python -m venv .venv
source .venv/bin/activate
pip install 'maturin>=1.15,<2' pytest scipy
maturin develop --release
cargo test --workspace --locked
pytest tests/
```

`maturin develop --release` compiles the workspace and installs the extension
into the active environment. Repeat it after changing Rust sources; the
Python side is plain source and needs no build step.

One mechanical detail worth knowing: the profiler releases the GIL while it
computes. Threads that call `profile` do not serialize behind the
interpreter, which matters if you profile several frames concurrently.